In [9]:
import numpy as np
import tensorflow as tf
from tf_pwa.config_loader import ConfigLoader 
import json
import os

In [10]:
# --- Configuration ---
CONFIG_FILE = "config_a.yml"
PARAMS_FILE = "final_params_full.json"

# Ensure we are in the correct directory (Analysis)
if not os.path.exists(CONFIG_FILE):
    print("Warning: config file not found in current directory.")

In [11]:
# 1. Load Configuration
config = ConfigLoader(CONFIG_FILE)

# 2. Load Parameters (to maintain system properties)
with open(PARAMS_FILE, 'r') as f:
    params_dict = json.load(f)['value']

In [12]:
# 3. Setup Kinematics (Hardcoded from tf_pwa_analysis_Gemini.py)
particles = list(config.get_decay().outs)
particle_map = {p.name: p for p in particles}

# Vectors matching Julia comparison
p4_dict = {
    particle_map["D"]: tf.constant([[2.0452, -0.1467, 0.2235, -0.7847]], dtype=tf.float64),
    particle_map["D0"]: tf.constant([[2.2606, 0.2284, -0.3689, 1.2019]], dtype=tf.float64),
    particle_map["K"]: tf.constant([[0.7718, -0.0873, 0.1803, -0.5584]], dtype=tf.float64),
    particle_map["pi"]: tf.constant([[0.2017, 0.0056, -0.0349, 0.1413]], dtype=tf.float64)
}

phsp_variables = config.data.cal_angle(p4_dict)
phsp_variables["c"] = np.array([-1.0]) # Extra variable

In [13]:
# 4. Set Parameters for Psi(4040)
# Isolate Psi(4040) (Chain Index 5)
# Chain: Bp -> Psi(4040) + K+; Psi(4040) -> D+ Dst0 (D0 pi0) .. wait, D+ Dst-?
# B -> D K Dx

amp_model = config.get_amplitude()
dg = amp_model.decay_group
all_chains = dg.chains

chain_idx = 5
chain = all_chains[chain_idx]

print(f"Selected Chain: {chain}")

# Set used chains to ONLY this one
dg.set_used_chains([chain_idx])

# Prepare Parameters: All zero except this chain's couplings = 1.0
p_new = params_dict.copy()

# Reset all couplings to 0
for k in p_new:
    if "total" in k or "g_ls" in k:
        if k.endswith("r") or k.endswith("i"):
            p_new[k] = 0.0

# Set Psi(4040) couplings to 1.0
# Iterate over step in chain to build prefix
for d_idx, d in enumerate(chain.chain):
    c_name = d.core.name.replace("(1+)", "(1.)")
    o_names = [p.name.replace("(1+)", "(1.)") for p in d.outs]
    prefix = f"{c_name}->{'.'.join(o_names)}"
    
    # Prod LS=0 (idx=0), Decay LS=0 (idx=0) for Psi(4040) [L=1, l=1]??
    # Wait, Psi(4040) is [L=1, l=1].
    # In tf_pwa_analysis_Gemini.py lines 80:
    # ("Psi(4040) [L=1, l=1]", 5, 0, 0)
    # So Prod_LS index is 0. Decay_LS index is 0.
    # Let's verify ls list length for Psi(4040).
    
    ls_idx = 0
    # Hardcoded 0 based on granular_groups definition
    
    for k in p_new:
        if prefix in k and ("total" in k or "g_ls" in k):
            if k.endswith(f"_{ls_idx}r"):
                print(f"Setting {k} = 1.0")
                p_new[k] = 1.0

config.set_params(p_new)


Selected Chain: [Bp->Psi(4040)+K, Psi(4040)->Dst+D, Dst->D0+pi]
Setting Bp->Psi(4040).KPsi(4040)->Dst.DDst->D0.pi_total_0r = 1.0
Setting Bp->Psi(4040).K_g_ls_0r = 1.0
Setting Bp->Psi(4040).KPsi(4040)->Dst.DDst->D0.pi_total_0r = 1.0
Setting Psi(4040)->Dst.D_g_ls_0r = 1.0
Setting Bp->X(3872).KX(3872)->Dst.DDst->D0.pi_total_0r = 1.0
Setting Dst->D0.pi_g_ls_0r = 1.0
Setting Bp->X(3915)(0-).KX(3915)(0-)->Dst.DDst->D0.pi_total_0r = 1.0
Setting Bp->chi(c2)(3930).Kchi(c2)(3930)->Dst.DDst->D0.pi_total_0r = 1.0
Setting Bp->X(3940)(1.).KX(3940)(1.)->Dst.DDst->D0.pi_total_0r = 1.0
Setting Bp->X(3993).KX(3993)->Dst.DDst->D0.pi_total_0r = 1.0
Setting Bp->Psi(4040).KPsi(4040)->Dst.DDst->D0.pi_total_0r = 1.0
Setting Bp->X(4300).KX(4300)->Dst.DDst->D0.pi_total_0r = 1.0
Setting Bp->NR(0-)SPp.KNR(0-)SPp->Dst.DDst->D0.pi_total_0r = 1.0
Setting Bp->NR(1.)PSp.KNR(1.)PSp->Dst.DDst->D0.pi_total_0r = 1.0
Setting Bp->NR(0-)SPm.KNR(0-)SPm->Dst.DDst->D0.pi_total_0r = 1.0
Setting Bp->NR(1-)PPm.KNR(1-)PPm->Dst.DDst

True

In [14]:
# 5. Calculate Amplitude
val = dg.get_amp(phsp_variables).numpy().flatten()[0]

print(f"\nCalculated Amplitude for Psi(4040):")
print(f"{val}\n")
print(f"Real: {val.real}")
print(f"Imag: {val.imag}")
print(f"Abs:  {abs(val)}")


Calculated Amplitude for Psi(4040):
(-0.0006049977354135942-0.0030870270680673287j)

Real: -0.0006049977354135942
Imag: -0.0030870270680673287
Abs:  0.0031457524344480677


In [15]:
#-0.0006049977354135942 - 0.0030870270680673287im

In [16]:
# --- D-Matrix Inspection ---
print("\n--- Inspecting D-Matrices ---\n")

# We use a monkey-patching approach to intercept the D-matrix calculation
# during the full amplitude evaluation. This ensures we see exactly what is used.

saved_d_matrices = []

# Identify the Decay class used (HelicityDecay)
# We can get it from the first chain/decay in the group
DecayClass = type(dg.chains[0][0])
original_get_D = DecayClass.get_D_matrix_term

def intercepted_get_D(self_decay, data, data_p, **kwargs):
    # Call original method
    ret = original_get_D(self_decay, data, data_p, **kwargs)
    
    # Capture the value for the first event
    val = ret.numpy()[0]
    shape = ret.shape
    name = str(self_decay)
    
    # Store it
    saved_d_matrices.append({
        "name": name,
        "value": val,
        "shape": shape
    })
    
    return ret

# Apply the patch
DecayClass.get_D_matrix_term = intercepted_get_D

print("Running amplitude calculation to capture matrices...")
try:
    # Trigger calculation
    dg.get_amp(phsp_variables)
except Exception as e:
    print(f"Calculation error: {e}")
finally:
    # Restore original method immediately
    DecayClass.get_D_matrix_term = original_get_D

print(f"\nCaptured {len(saved_d_matrices)} D-matrices.\n")

# Print results
for i, item in enumerate(saved_d_matrices):
    print(f"Decay [{i}]: {item['name']}")
    print(f"  Shape: {item['shape']}")
    print(f"  Value (idx 0):\n{item['value']}")
    print("-" * 30)


--- Inspecting D-Matrices ---

Running amplitude calculation to capture matrices...

Captured 3 D-matrices.

Decay [0]: Bp->Psi(4040)+K
  Shape: (1, 1, 3, 1)
  Value (idx 0):
[[[0.+0.j]
  [1.+0.j]
  [0.+0.j]]]
------------------------------
Decay [1]: Psi(4040)->Dst+D
  Shape: (1, 3, 3, 1)
  Value (idx 0):
[[[-4.07423259e-01-9.12918336e-01j]
  [-9.86821650e-03-2.21118348e-02j]
  [-1.19509251e-04-2.67785857e-04j]]

 [[-2.42139409e-02+0.00000000e+00j]
  [ 9.99413513e-01+0.00000000e+00j]
  [ 2.42139409e-02+0.00000000e+00j]]

 [[-1.19509251e-04+2.67785857e-04j]
  [ 9.86821650e-03-2.21118348e-02j]
  [-4.07423259e-01+9.12918336e-01j]]]
------------------------------
Decay [2]: Dst->D0+pi
  Shape: (1, 3, 1, 1)
  Value (idx 0):
[[[ 0.12378949+0.33405639j]]

 [[-0.86380842+0.j        ]]

 [[-0.12378949+0.33405639j]]]
------------------------------


In [20]:
print(-4.07423259e-01-9.12918336e-01j)
print(-9.86821650e-03-2.21118348e-02j)
print(-1.19509251e-04-2.67785857e-04j)

print(-2.42139409e-02+0.00000000e+00j)
print(9.99413513e-01+0.00000000e+00j)
print(2.42139409e-02+0.00000000e+00j)

print(-1.19509251e-04+2.67785857e-04j)
print(9.86821650e-03-2.21118348e-02j)
print(-4.07423259e-01+9.12918336e-01j)

print()
print(0.12378949+0.33405639j)
print(-0.86380842+0.j)
print(-0.12378949+0.33405639j)

(-0.407423259-0.912918336j)
(-0.0098682165-0.0221118348j)
(-0.000119509251-0.000267785857j)
(-0.0242139409+0j)
(0.999413513+0j)
(0.0242139409+0j)
(-0.000119509251+0.000267785857j)
(0.0098682165-0.0221118348j)
(-0.407423259+0.912918336j)

(0.12378949+0.33405639j)
(-0.86380842+0j)
(-0.12378949+0.33405639j)
